# Product Demand Forecasting

This notebook describes **product demand forecasting** using Amazon product data. Since we don't have historical sales time series, we use **rating count** as a proxy for demand: products with more ratings are assumed to have higher demand.

**Steps:**
1. Load and clean the data  
2. Explore the data (EDA)  
3. Engineer features (price, category, text length)  
4. Train a model to predict demand (rating count)  
5. Evaluate and interpret results

## 1. Load and clean data

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data, get_analysis_sample
from src.feature_engineering import add_demand_level, add_price_features, add_category_features, add_text_length_features, build_feature_matrix
from src.model import train_demand_model, get_feature_importance
from src.config import TARGET_COLUMN, DEMAND_LEVEL_LABELS

In [ ]:
df_raw = load_data()
print("Shape:", df_raw.shape)
df_raw.head()

In [ ]:
# Use a sample for faster analysis (optional: use df_raw for full data)
df = get_analysis_sample(df_raw, max_rows=3000)
print("Analysis sample shape:", df.shape)
df[["discounted_price", "actual_price", "discount_percentage", "rating", "rating_count"]].describe()

## 2. Exploratory Data Analysis

In [ ]:
df_eda = add_demand_level(df)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

df_eda["rating_count"].hist(bins=50, ax=axes[0, 0], edgecolor="black", alpha=0.7)
axes[0, 0].set_title("Distribution of Demand (rating_count)")
axes[0, 0].set_xlabel("Rating count")

df_eda["demand_level"].value_counts().sort_index().plot(kind="bar", ax=axes[0, 1], color="steelblue")
axes[0, 1].set_title("Demand level counts")
axes[0, 1].set_xlabel("Demand level")

df_eda.plot.scatter(x="discounted_price", y="rating_count", alpha=0.4, ax=axes[1, 0])
axes[1, 0].set_title("Price vs Demand")

df_eda.boxplot(column="rating_count", by="demand_level", ax=axes[1, 1])
axes[1, 1].set_title("Demand by level")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
df_eda = add_category_features(df_eda)
top_cats = df_eda["category_top"].value_counts().head(10)
top_cats.plot(kind="barh", figsize=(8, 5))
plt.title("Top 10 categories by product count")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

## 3. Feature engineering and model

In [ ]:
df = add_price_features(df)
df = add_category_features(df)
df = add_text_length_features(df)
X, y, feature_names = build_feature_matrix(df)
print("Features:", feature_names)
print("Samples:", len(X))

In [ ]:
model, X_test, y_test, y_pred, metrics = train_demand_model(X, y)
print("Demand forecasting metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
importance = get_feature_importance(model, feature_names)
importance.plot(kind="barh", figsize=(8, 5))
plt.title("Feature importance for demand forecast")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 4. Summary

- **Demand** is proxied by `rating_count` (more ratings ≈ more demand).
- **Features** such as price, discount, rating, category, and description length help predict demand.
- The **Random Forest** model gives MAE, RMSE, and R²; feature importance shows which factors drive demand most.
- For true time-series demand forecasting, historical sales data would be needed; this project illustrates the approach with the available Amazon dataset.